# HW4 — Jefferson Li

Run with the course .venv. See [README](README.md) for sources and the full test.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "Lectures/modeling/course_data.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from inside the course repository.")

modeling = repo / "Lectures/modeling"
sys.path.insert(0, str(modeling))
from course_data import (
    apply_membership, load_data, load_membership, make_labels,
    prediction_metrics, price_features,
)

database = modeling / "data/equities_class.duckdb"
prices, _, universe, benchmark = load_data(database, end="2015-12-31")
calendar = pd.DatetimeIndex(benchmark["date"])
membership = load_membership(database, end="2015-12-31")

## Features and targets

Use 63-session momentum and volatility at the signal close. The target spans next-session entry to the first market close on or after 90 calendar days.

In [2]:
features = ["momentum_63", "volatility_63"]
panel = price_features(prices, calendar)
panel = apply_membership(panel, membership, calendar)
labeled = make_labels(panel, prices, calendar, holding_days=90, max_exit_delay=0)

development = labeled.loc[
    labeled["date"].between("2010-01-01", "2015-12-31") & labeled["eligible"]
].copy()
assert np.isfinite(development[features].to_numpy()).all()

completed = development.loc[development["target"].notna()]
assert completed["date"].lt(completed["entry_date"]).all()
assert completed["entry_date"].add(pd.Timedelta(days=90)).eq(completed["scheduled_exit"]).all()
expected_exits = calendar[calendar.searchsorted(completed["scheduled_exit"])]
assert completed["label_end"].eq(expected_exits).all()

In [3]:
amt = development.loc[
    development["symbol"].eq("AMT") & development["date"].eq("2010-04-09")
]
assert len(amt) == 1
trade = amt.iloc[0]
assert trade["entry_date"] == pd.Timestamp("2010-04-12")
assert trade["scheduled_exit"] == pd.Timestamp("2010-07-11")
assert trade["label_end"] == pd.Timestamp("2010-07-12")
assert np.isclose(trade["target"], 34.60 / 32.34 - 1)

audit_columns = [
    "symbol", "date", "entry_date", "entry_price",
    "scheduled_exit", "label_end", "exit_price", "target",
]
audit = amt[audit_columns].rename(columns={
    "date": "signal_date", "scheduled_exit": "maturity_date", "label_end": "exit_date",
})
display(audit.assign(target=audit["target"].map("{:+.2%}".format)).reset_index(drop=True))

,symbol,signal_date,entry_date,entry_price,maturity_date,exit_date,exit_price,target
0,AMT,2010-04-09,2010-04-12,32.34,2010-07-11,2010-07-12,34.6,+6.99%


## Validation

Each fold fits on trades closed before its first signal. RMSE is in percentage points.

In [4]:
summaries = []
score_frames = []

for year in (2013, 2014):
    validation = development.loc[
        development["date"].between(f"{year}-01-01", f"{year}-12-31")
    ].copy()
    first_signal = validation["date"].min()
    earlier = development["date"].lt(first_signal)
    train = development.loc[
        earlier & development["label_end"].lt(first_signal) & development["target"].notna()
    ]
    assert len(train) and len(validation)
    assert train["date"].max() < first_signal
    assert train["label_end"].max() < first_signal

    model = make_pipeline(StandardScaler(), Ridge(alpha=100))
    model.fit(train[features], train["target"])

    scores = validation[["date", "symbol"]].copy()
    scores["prediction"] = model.predict(validation[features])
    score_frames.append(scores)
    evaluation = scores.join(validation["target"]).dropna(subset=["target"])
    constant = np.full(len(evaluation), train["target"].mean())
    metrics = prediction_metrics(evaluation, evaluation["prediction"])

    summaries.append({
        "validation_year": year,
        "first_signal": first_signal.date(),
        "latest_train_exit": train["label_end"].max().date(),
        "train_rows": len(train),
        "ridge_RMSE_pp": 100 * metrics["RMSE"],
        "mean_RMSE_pp": 100 * root_mean_squared_error(evaluation["target"], constant),
        "mean_daily_rank_IC": metrics["mean_daily_rank_IC"],
    })

validation_scores = pd.concat(score_frames, ignore_index=True)
validation_summary = pd.DataFrame(summaries).set_index("validation_year")
display(validation_summary.round(3))

,first_signal,latest_train_exit,train_rows,ridge_RMSE_pp,mean_RMSE_pp,mean_daily_rank_IC
validation_year,,,,,,
2013,2013-01-02,2012-12-31,20034,14.019,14.402,0.008
2014,2014-01-02,2013-12-31,27284,14.337,14.136,-0.035


## Class backtest

Replay the ledger. The first BESS sale is partial because daily turnover caps the fill.

In [5]:
from hw4_backtest import audit, run_backtest

history, fills, open_lots, skipped = run_backtest(
    validation_scores, prices, calendar, "2013-01-02", "2014-12-31",
)
audit(history, fills, open_lots, prices, calendar)
first = fills.loc[fills["lot_id"].eq(fills.iloc[0]["lot_id"])]
example = first.iloc[:2]
display(example[["side", "symbol", "date", "price", "units", "fee", "cash_change"]].round(2))
display(history.loc[example["date"], ["cash", "position_value", "equity"]].round(2))

,side,symbol,date,price,units,fee,cash_change
0,BUY,BESS,2013-01-03,68.6,36.25,2.49,-2489.24
2,SELL,BESS,2013-04-03,77.0,9.90,0.76,761.54


,cash,position_value,equity
date,,,
2013-01-03,997510.76,2486.75,999997.51
2013-04-03,996615.15,4146.45,1000761.60
